# 05 — Auditoría de resultados HAI22.04

**Protocolo:** análisis exclusivamente de archivos existentes. No se entrenan modelos ni se modifican los CSV originales. Las columnas se inspeccionan antes de utilizarlas. Las métricas no se reinterpretan automáticamente sin conocer su definición.

In [1]:
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt
ROOT = Path(r"C:\Users\hugoc\Local\PUCV\Doctorado\Cursos\Semestre-2-2026\Industria-Inteligente\archive\resultados_hai_arboles")
assert ROOT.is_dir(), f"No existe carpeta: {ROOT}"
DEST = ROOT / 'analisis'
DEST.mkdir(exist_ok=True)
FILES = sorted(p for p in ROOT.rglob('*.csv') if DEST not in p.parents)
print('CSV encontrados:', len(FILES))
def load_csv(p):
    for enc in ('utf-8-sig','latin1'):
        try:
            df=pd.read_csv(p,encoding=enc,sep=None,engine='python')
            return df
        except UnicodeDecodeError: continue
    raise ValueError(f'No se pudo leer {p}')
def inventory():
    rows=[]
    for p in FILES:
        try:
            df=load_csv(p)
            rows.append(dict(archivo=str(p.relative_to(ROOT)),filas=len(df),columnas=len(df.columns),nombres_columnas=' | '.join(map(str,df.columns)),nulos=int(df.isna().sum().sum()),error=''))
        except Exception as e:
            rows.append(dict(archivo=str(p.relative_to(ROOT)),filas=None,columnas=None,nombres_columnas='',nulos=None,error=repr(e)))
    return pd.DataFrame(rows)


CSV encontrados: 86


In [2]:
inv=inventory()
inv.to_csv(DEST/'05_inventario.csv',index=False,encoding='utf-8-sig')
display(inv)


,archivo,filas,columnas,nombres_columnas,nulos,error
0,auditoria\inventario.csv,10,10,archivo | filas | columnas | faltantes | inici...,0,
1,auditoria\salidas.csv,24,2,componente | salida,0,
2,metricas\comparacion_deteccion.csv,189,9,metodo | config | archivo | n | n_alerta | fra...,378,
3,metricas\deteccion_DT_FULL.csv,9,9,metodo | config | archivo | n | n_alerta | fra...,18,
4,metricas\deteccion_DT_SIN_P1.csv,9,9,metodo | config | archivo | n | n_alerta | fra...,18,
...,...,...,...,...,...,...
81,metricas\seleccion_XGB_FULL.csv,24,7,metodo | configuracion | variable | componente...,0,
82,metricas\seleccion_XGB_SIN_P1.csv,24,7,metodo | configuracion | variable | componente...,0,
83,metricas\seleccion_XGB_SIN_P2.csv,24,7,metodo | configuracion | variable | componente...,0,
84,metricas\seleccion_XGB_SIN_P3.csv,24,7,metodo | configuracion | variable | componente...,0,


In [3]:
import re
expected=[f'{m}_{a}' for m in ('DT','RF','ET','XGB') for a in ('FULL','SIN_P1','SIN_P2','SIN_P3','SIN_P4')]
rows=[]
for config in expected+['PERSIST_FULL']:
    matches=[p for p in FILES if re.search(r'(?<![A-Za-z0-9])'+re.escape(config)+r'(?![A-Za-z0-9])',p.stem,re.I)]
    rows.append({'configuracion':config,'n_archivos':len(matches),'archivos':' | '.join(str(p.relative_to(ROOT)) for p in matches)})
coverage=pd.DataFrame(rows)
coverage.to_csv(DEST/'05_cobertura_configuraciones.csv',index=False,encoding='utf-8-sig')
display(coverage)
print('AVISO: la ausencia de coincidencia por nombre NO demuestra ausencia de resultados.')


,configuracion,n_archivos,archivos
0,DT_FULL,4,metricas\deteccion_DT_FULL.csv | metricas\epis...
1,DT_SIN_P1,4,metricas\deteccion_DT_SIN_P1.csv | metricas\ep...
2,DT_SIN_P2,4,metricas\deteccion_DT_SIN_P2.csv | metricas\ep...
3,DT_SIN_P3,4,metricas\deteccion_DT_SIN_P3.csv | metricas\ep...
4,DT_SIN_P4,4,metricas\deteccion_DT_SIN_P4.csv | metricas\ep...
5,RF_FULL,4,metricas\deteccion_RF_FULL.csv | metricas\epis...
6,RF_SIN_P1,4,metricas\deteccion_RF_SIN_P1.csv | metricas\ep...
7,RF_SIN_P2,4,metricas\deteccion_RF_SIN_P2.csv | metricas\ep...
8,RF_SIN_P3,4,metricas\deteccion_RF_SIN_P3.csv | metricas\ep...
9,RF_SIN_P4,4,metricas\deteccion_RF_SIN_P4.csv | metricas\ep...


AVISO: la ausencia de coincidencia por nombre NO demuestra ausencia de resultados.


In [5]:

checks = []

for p in FILES:
    try:
        df = load_csv(p)

        checks.append({
            'archivo': str(p.relative_to(ROOT)),
            'duplicados': int(df.duplicated().sum()),
            'columnas_duplicadas': str(
                df.columns[df.columns.duplicated()].tolist()
            ),
            'columnas_totalmente_nulas': ' | '.join(
                df.columns[df.isna().all()].astype(str)
            ),
            'columnas_test': ' | '.join(
                str(c) for c in df.columns
                if 'test' in str(c).lower()
            ),
            'columnas_metricas_candidatas': ' | '.join(
                str(c) for c in df.columns
                if any(
                    w in str(c).lower()
                    for w in (
                        'mae', 'rmse', 'mse', 'precision',
                        'recall', 'f1', 'auc',
                        'threshold', 'umbral', 'score'
                    )
                )
            )
        })

    except Exception as e:
        checks.append({
            'archivo': str(p.relative_to(ROOT)),
            'error': str(e)
        })

checks = pd.DataFrame(checks)

checks.to_csv(
    DEST / '05_calidad.csv',
    index=False,
    encoding='utf-8-sig'
)

display(checks)

resumen = (
    f'CSV: {len(FILES)}\n'
    f'Archivos con error: {int(inv.error.astype(bool).sum())}\n'
    f'Configuraciones sin coincidencia nominal: '
    f'{coverage.loc[coverage.n_archivos.eq(0), "configuracion"].tolist()}\n'
    'Revisar manualmente nombres, definición de métricas, '
    'partición temporal y cobertura TEST antes de interpretar resultados.\n'
)

(DEST / '05_resumen.txt').write_text(
    resumen,
    encoding='utf-8'
)


,archivo,duplicados,columnas_duplicadas,columnas_totalmente_nulas,columnas_test,columnas_metricas_candidatas
0,auditoria\inventario.csv,0,[],,,
1,auditoria\salidas.csv,0,[],,,
2,metricas\comparacion_deteccion.csv,0,[],,,
3,metricas\deteccion_DT_FULL.csv,0,[],,,
4,metricas\deteccion_DT_SIN_P1.csv,0,[],,,
...,...,...,...,...,...,...
81,metricas\seleccion_XGB_FULL.csv,0,[],,,rmse_selection | mae_selection
82,metricas\seleccion_XGB_SIN_P1.csv,0,[],,,rmse_selection | mae_selection
83,metricas\seleccion_XGB_SIN_P2.csv,0,[],,,rmse_selection | mae_selection
84,metricas\seleccion_XGB_SIN_P3.csv,0,[],,,rmse_selection | mae_selection


197